# Ensemble Analysis and Prediction Agreement Study

This notebook presents a systematic analysis of prediction agreement,
disagreement, and complementarity across multiple Transformer-based
models trained for the news article classification task.

The objective is not direct performance optimization, but to provide
empirical evidence supporting the ensemble and hard-case routing
strategies adopted in the final submissions.

---

## Models Considered

We analyze **16 fine-tuned Transformer models**, covering:

- **Architectures**
  - DeBERTa
  - RoBERTa

- **Input lengths**
  - Max length 512
  - Max length 256

- **Random seeds**
  - 42
  - 1337
  - 2024
  - No-seed (single deterministic run)

All models generate predictions over the **20,000 evaluation samples**,
loaded from submission-format CSV files to ensure full consistency with
leaderboard outputs.

---

## Results and Empirical Findings

### 1. Global Prediction Overlap

Across all model pairs, prediction overlap is consistently high but
non-trivial:

- **Global agreement ranges approximately from 0.85 to 0.92**
- The highest overlap is observed between models sharing both
  architecture and input length
- Cross-architecture agreement (DeBERTa vs RoBERTa) is systematically
  lower, indicating meaningful architectural diversity

This confirms that the model pool is neither redundant nor independent,
a desirable condition for effective ensembling.

---

### 2. Intra-Model Stability Across Seeds

Models trained with different random seeds exhibit strong stability:

- **Intra-seed overlap**
  - DeBERTa (512): ~0.87–0.90
  - DeBERTa (256): ~0.86–0.91
  - RoBERTa (512): ~0.87–0.92
  - RoBERTa (256): ~0.91–0.92

This indicates limited sensitivity to stochastic initialization and
suggests that seed ensembling provides diminishing returns compared to
architecture-level diversity.

---

### 3. Cross-Architecture Agreement

Agreement between DeBERTa and RoBERTa models is consistently lower than
intra-architecture agreement:

- **Cross-architecture overlap typically lies in the 0.85–0.89 range**
- Pairwise disagreement between architectures often exceeds **12–15%**

This behavior motivates architecture-aware ensemble strategies and
confirms that RoBERTa and DeBERTa capture partially distinct decision
boundaries.

---

### 4. Hard-Case Identification via Disagreement
**Note on Hard-Case Definition Used in the Paper**

While multiple definitions of hard cases are explored in this notebook for
diagnostic and routing purposes, the experimental analysis reported in the
paper adopts a simpler and more inclusive definition of hard cases, as
implemented in `Heatmap_figures.ipynb`.

In particular, a sample is considered a hard case if **at least two
Transformer models disagree in their predicted label**.
Under this definition, **6,839 samples out of 20,000 (≈34.2%)** are classified
as hard cases.

This broader criterion is used exclusively for analyzing cross-architecture
prediction overlap and model complementarity, and should not be interpreted
as a measure of extreme ambiguity.
More restrictive definitions based on higher disagreement thresholds or
vote entropy are used only for selective routing strategies and are not
reported in the paper.
**Here, in this code we use another definition, just to better understand the problem.**


A per-sample disagreement score is computed across all 16 models.
Using a conservative threshold of **≥ 40% disagreement**, we identify:

- **1,338 hard cases out of 20,000 samples**
- Corresponding to **≈ 6.7% of the evaluation set**

These samples concentrate most of the model uncertainty and are natural
candidates for selective routing strategies.

The most ambiguous samples exhibit disagreement rates up to **75%**,
indicating near-complete architectural disagreement.

---

### 5. Hard-Case Distribution by Predicted Class

Analyzing hard cases by predicted label (using a fixed RoBERTa backbone)
reveals an uneven distribution:

- **Entertainment**: ~27%
- **General News**: ~26%
- **International**: ~21%
- **Business**: ~12%
- Remaining classes (Technology, Health, Sports) jointly account for
  less than 15%

This suggests that topic ambiguity and editorial overlap play a major
role in driving model disagreement.

---

### 6. Majority Vote Baseline

A majority-vote ensemble across all models is generated as a low-variance
baseline. While robust, this approach does not exploit:
- confidence differences,
- architectural specialization,
- or uncertainty-aware routing,

and therefore serves primarily as a diagnostic reference rather than
the final submission strategy.

---

## Summary

Overall, the analysis shows that:
- Transformer models are **highly stable but not redundant**,
- architectural diversity is the dominant source of useful disagreement,
- a **small subset of samples (~7%)** concentrates most uncertainty,
- these findings directly justify the use of **logit-level ensembles**
  and **hard-case routing strategies** in the final system.

All ensemble configurations used in the final submissions are grounded
in the empirical evidence reported in this notebook.




In [29]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [30]:
files = {
	# ===== DeBERTa 512 =====
	"deberta_42":     "deberta_MAXLEN512/deberta_seed_42_processed/submission_seed42.csv",
	"deberta_1337":   "deberta_MAXLEN512/deberta_seed_1337_processed/submission_seed_daberta_processed1337.csv",
	"deberta_2024":   "deberta_MAXLEN512/deberta_seed_2024_processed/submission_seed_daberta_processed2024.csv",
	"deberta_noseed": "deberta_MAXLEN512/submission_deberta_processed_noseed_MAXLEN_512.csv",

	# ===== DeBERTa 256 =====
	"deberta256_noseed": "deberta_MAXLEN256/submission_deberta_processed_noseed_MAXLEN_256.csv",
    "deberta256_42" :  "deberta_MAXLEN256/deberta_seed42_MAXLEN256\submission_roberta_processed_seed42_MAXLEN256.csv",
    "deberta256_1337" : "deberta_MAXLEN256/deberta_seed1337_MAXLEN256/submission_roberta_processed_seed1337_MAXLEN256.csv",
    "deberta256_2024" : "deberta_MAXLEN256/deberta_seed2024_MAXLEN256/submission_roberta_processed_seed2024_MAXLEN256.csv",

	# ===== RoBERTa 512 =====
	"roberta_42":     "roberta_MAXLEN512/roberta_processed_seed_42/submission_seed_roberta_processed42.csv",
	"roberta_1337":   "roberta_MAXLEN512/roberta_processed_seed_1337/submission_seed_roberta_processed1337.csv",
	"roberta_2024":   "roberta_MAXLEN512/roberta_processed_seed_2024/submission_seed_roberta_processed2024.csv",
	"roberta_noseed": "roberta_MAXLEN512/submission_roberta_processed_noseed_MAXLEN_512.csv",

	# ===== RoBERTa 256 =====
	"roberta256_42":   "roberta_MAXLEN256/roberta_seed42_MAXLEN256/submission_roberta_processed_seed42_MAXLEN256.csv",
	"roberta256_1337": "roberta_MAXLEN256/roberta_seed1337_MAXLEN256/submission_roberta_processed_seed1337_MAXLEN256.csv",
	"roberta256_2024": "roberta_MAXLEN256/roberta_seed2024_MAXLEN256/submission_roberta_processed_seed2024_MAXLEN256.csv",
    "roberta_256_noseed" : "roberta_MAXLEN256/submission_roberta_processed_noseed_MAXLEN_256.csv"
}


In [31]:
ARTIFACTS_DIR = "analysis_artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

In [32]:
preds = {}
ids_ref = None

for name, path in files.items():
	df = pd.read_csv(path)
	assert {"Id", "Predicted"}.issubset(df.columns)

	if ids_ref is None:
		ids_ref = df["Id"].values
	else:
		assert np.all(df["Id"].values == ids_ref), f"ID mismatch in {name}"

	preds[name] = df["Predicted"].values

print(f"Loaded {len(preds)} models | {len(ids_ref)} samples")


Loaded 16 models | 20000 samples


In [33]:

# OVERLAP MATRIX (GLOBAL)


model_names = list(preds.keys())
n_models = len(model_names)

overlap = pd.DataFrame(
	index=model_names,
	columns=model_names,
	dtype=float
)

for m1 in model_names:
	for m2 in model_names:
		overlap.loc[m1, m2] = np.mean(preds[m1] == preds[m2])

overlap = overlap.astype(float)

print("\n=== Global Overlap Matrix ===")
print(overlap.round(4))



=== Global Overlap Matrix ===
                    deberta_42  deberta_1337  deberta_2024  deberta_noseed  \
deberta_42              1.0000        0.8982        0.8966          0.8743   
deberta_1337            0.8982        1.0000        0.8956          0.8713   
deberta_2024            0.8966        0.8956        1.0000          0.8692   
deberta_noseed          0.8743        0.8713        0.8692          1.0000   
deberta256_noseed       0.8690        0.8681        0.8692          0.8730   
deberta256_42           0.9011        0.8952        0.8956          0.8652   
deberta256_1337         0.8886        0.8890        0.8871          0.8518   
deberta256_2024         0.8968        0.8902        0.9044          0.8610   
roberta_42              0.8839        0.8810        0.8813          0.8555   
roberta_1337            0.8790        0.8834        0.8779          0.8549   
roberta_2024            0.8782        0.8744        0.8834          0.8548   
roberta_noseed          0.8596   

In [14]:
OVERLAP_TXT = f"{ARTIFACTS_DIR}/overlap_matrix_fixed.txt"
overlap.round(6).to_csv(OVERLAP_TXT, sep="\t")
print(f"Saved overlap matrix to {OVERLAP_TXT}")


Saved overlap matrix to analysis_artifacts/overlap_matrix_fixed.txt


In [ ]:

# INTRA-MODEL OVERLAP (SEEDS)

def group_by_prefix(names):
	groups = {}
	for n in names:
		prefix = n.split("_")[0]   # deberta, roberta
		groups.setdefault(prefix, []).append(n)
	return groups

groups = group_by_prefix(model_names)

for g, models in groups.items():
	if len(models) < 2:
		continue
	print(f"\n=== Intra-model overlap: {g} ===")
	sub = overlap.loc[models, models]
	print(sub.round(4))



=== Intra-model overlap: deberta ===
                deberta_42  deberta_1337  deberta_2024  deberta_noseed
deberta_42          1.0000        0.8982        0.8966          0.8743
deberta_1337        0.8982        1.0000        0.8956          0.8713
deberta_2024        0.8966        0.8956        1.0000          0.8692
deberta_noseed      0.8743        0.8713        0.8692          1.0000

=== Intra-model overlap: deberta256 ===
                   deberta256_noseed  deberta256_42  deberta256_1337  \
deberta256_noseed             1.0000         0.8630           0.8522   
deberta256_42                 0.8630         1.0000           0.9067   
deberta256_1337               0.8522         0.9067           1.0000   
deberta256_2024               0.8655         0.9110           0.9042   

                   deberta256_2024  
deberta256_noseed           0.8655  
deberta256_42               0.9110  
deberta256_1337             0.9042  
deberta256_2024             1.0000  

=== Intra-model ove

In [15]:

# CROSS-ARCHITECTURE OVERLAP


deberta_models = [m for m in model_names if "deberta" in m]
roberta_models = [m for m in model_names if "roberta" in m]

cross_arch = overlap.loc[deberta_models, roberta_models]

print("\n=== Cross-Architecture Overlap ===")
print(cross_arch.round(4))

cross_arch.round(6).to_csv(
	f"{ARTIFACTS_DIR}/overlap_cross_arch.txt",
	sep="\t"
)



=== Cross-Architecture Overlap ===
                   roberta_42  roberta_1337  roberta_2024  roberta_noseed  \
deberta_42             0.8839        0.8790        0.8782          0.8596   
deberta_1337           0.8810        0.8834        0.8744          0.8578   
deberta_2024           0.8813        0.8779        0.8834          0.8588   
deberta_noseed         0.8555        0.8549        0.8548          0.8522   
deberta256_noseed      0.8538        0.8536        0.8528          0.8488   
deberta256_42          0.8888        0.8860        0.8834          0.8530   
deberta256_1337        0.8822        0.8780        0.8730          0.8414   
deberta256_2024        0.8866        0.8783        0.8845          0.8514   

                   roberta256_42  roberta256_1337  roberta256_2024  \
deberta_42                0.8845           0.8776           0.8806   
deberta_1337              0.8812           0.8814           0.8784   
deberta_2024              0.8766           0.8779           

In [23]:

# HARD CASE ANALYSIS


pred_matrix = np.vstack([preds[m] for m in model_names])  # (n_models, n_samples)

def disagreement_rate(col):
	values, counts = np.unique(col, return_counts=True)
	return 1 - counts.max() / counts.sum()

disagreement = np.apply_along_axis(disagreement_rate, 0, pred_matrix)

hard_df = pd.DataFrame({
	"Id": ids_ref,
	"disagreement": disagreement
})

hard_df = hard_df.sort_values("disagreement", ascending=False)

print("\nTop hard cases:")
print(hard_df.head(10))



Top hard cases:
          Id  disagreement
9394    9394        0.7500
5922    5922        0.6875
14919  14919        0.6875
15273  15273        0.6875
11817  11817        0.6250
12017  12017        0.6250
11241  11241        0.6250
11960  11960        0.6250
19728  19728        0.6250
19851  19851        0.6250


In [24]:
HARD_THR = 0.4   # >=40% disagreement
hard_cases = hard_df[hard_df["disagreement"] >= HARD_THR]

print(f"\nHard cases: {len(hard_cases)} / {len(ids_ref)}")
hard_cases.to_csv(
	f"{ARTIFACTS_DIR}/hard_cases.csv",
	index=False
)



Hard cases: 1338 / 20000


In [25]:

# PAIRWISE DISAGREEMENT

pairwise_dis = pd.DataFrame(
	index=model_names,
	columns=model_names,
	dtype=float
)

for i, m1 in enumerate(model_names):
	for j, m2 in enumerate(model_names):
		if i == j:
			pairwise_dis.loc[m1, m2] = 0.0
		else:
			pairwise_dis.loc[m1, m2] = np.mean(preds[m1] != preds[m2])

pairwise_dis = pairwise_dis.astype(float)

print("\n=== Pairwise Disagreement ===")
print(pairwise_dis.round(4))

pairwise_dis.round(6).to_csv(f"{ARTIFACTS_DIR}/pairwise_disagreement.txt", sep="\t")



=== Pairwise Disagreement ===
                    deberta_42  deberta_1337  deberta_2024  deberta_noseed  \
deberta_42              0.0000        0.1018        0.1034          0.1257   
deberta_1337            0.1018        0.0000        0.1044          0.1287   
deberta_2024            0.1034        0.1044        0.0000          0.1308   
deberta_noseed          0.1257        0.1287        0.1308          0.0000   
deberta256_noseed       0.1310        0.1319        0.1308          0.1270   
deberta256_42           0.0989        0.1048        0.1044          0.1348   
deberta256_1337         0.1114        0.1110        0.1129          0.1482   
deberta256_2024         0.1032        0.1098        0.0956          0.1390   
roberta_42              0.1161        0.1190        0.1187          0.1445   
roberta_1337            0.1210        0.1166        0.1221          0.1451   
roberta_2024            0.1218        0.1257        0.1166          0.1452   
roberta_noseed          0.1404   

In [26]:

# MAJORITY VOTE (BASELINE ENSEMBLE)

from scipy.stats import mode

maj_vote, _ = mode(pred_matrix, axis=0, keepdims=False)

maj_vote = maj_vote.astype(int)

maj_df = pd.DataFrame({
	"Id": ids_ref,
	"Predicted": maj_vote
})

maj_df.to_csv(f"{ARTIFACTS_DIR}/submission_majority_vote.csv", index=False)


In [27]:
n_total = len(ids_ref)
n_hard  = len(hard_cases)

print(f"Hard cases: {n_hard} / {n_total} ({n_hard / n_total:.2%})")


Hard cases: 1338 / 20000 (6.69%)


In [28]:

# HARD CASES PER CLASS



BACKBONE = "roberta_noseed"

hard_with_pred = hard_cases.copy()
hard_with_pred["Predicted"] = preds[BACKBONE][hard_with_pred.index]

label_map = {
	0: "International",
	1: "Business",
	2: "Technology",
	3: "Entertainment",
	4: "Sports",
	5: "General",
	6: "Health"
}

pred_counts = hard_with_pred["Predicted"].value_counts().sort_index()
pred_perc   = pred_counts / pred_counts.sum()

summary = pd.DataFrame({
	"class": [label_map[i] for i in pred_counts.index],
	"count": pred_counts.values,
	"percentage": pred_perc.values
})

print(summary)



           class  count  percentage
0  International    277    0.207025
1       Business    158    0.118087
2     Technology     83    0.062033
3  Entertainment    366    0.273543
4         Sports     44    0.032885
5        General    352    0.263079
6         Health     58    0.043348
